<a href="https://colab.research.google.com/github/Rouba-Os/AI-Course/blob/main/Project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==================
# Import Libraries
# ==================
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# ==========
# Load Data
# ==========
stocks = pd.read_csv('historical_stocks.csv')
prices = pd.read_csv('historical_stock_prices.csv')

print("Prices Dataset: First 5 Rows")
print(prices.head())

print("\nStocks Dataset: First 5 Rows")
print(stocks.head())


In [ ]:
# =================
# Check Data Types
# =================
print("Prices Dataset: Info")
print(prices.info())

print("\nStocks Dataset: Info")
print(stocks.info())


In [ ]:
# =============================
# Convert date and set as index
# =============================
prices['date'] = pd.to_datetime(prices['date'])
prices.set_index('date', inplace=True)

print("Date Converted and Set as Index")
print(prices.head())


In [ ]:
# =============
# Create Year
# =============
prices['Year'] = prices.index.year

print("Year Column Added")
print(prices[['ticker','Year']].head())

In [ ]:
# =====================
# Check Missing Values
# =====================

print("Missing Values in Prices Dataset")
print(prices.isnull().sum())

print("\nMissing Values in Stocks Dataset")
print(stocks.isnull().sum())

In [ ]:
# ======================
# Handle Missing Values
# ======================

# Sort data first
prices = prices.sort_values(by=['ticker', 'date'])
print("Data Sorted by Ticker and Date")
print(prices.head())


# Interpolate price columns (per stock)
prices[['open','high','low','close','adj_close']] = (
prices.groupby('ticker')[['open','high','low','close','adj_close']]
.transform(lambda x: x.interpolate())
)

print("\nAfter Interpolation - Price Columns")
print(prices[['open','high','low','close']].head())



In [ ]:
# Handle volume column missing values
prices['volume'] = prices.groupby(['ticker','Year'])['volume'].transform(
    lambda x: x.fillna(x.median())
)

print("After Filling Volume With Median by Ticker and Year")
print(prices[['ticker','Year','volume']].head())
print()
print(prices.isnull().sum())


In [ ]:
# Handle Missing Values In Stocks Dataset
stocks['sector'] = stocks['sector'].fillna('Unknown')
stocks['industry'] = stocks['industry'].fillna('Unknown')

print("\nAfter Handling Missing Values - Stocks Dataset")
print(stocks.isnull().sum())

In [ ]:
# =====================
# Check Duplicates
# =====================
print("Duplicate Rows in Prices Dataset")
print(prices.duplicated().sum())

print("\nDuplicate Rows in Stocks Dataset")
print(stocks.duplicated().sum())

In [ ]:
# Remove Duplicates
prices.drop_duplicates(inplace=True)
stocks.drop_duplicates(inplace=True)

print("Prices Dataset After Removing Duplicates")
print("Shape:", prices.shape)


print("\nStocks Dataset After Removing Duplicates")
print("Shape:", stocks.shape)


In [ ]:
# =======================
# Create Decade Feature
# =======================
prices['Decade'] = (prices['Year'] // 10) * 10

print("Data with Decade Column")
print(prices[['ticker','Year','Decade']].head())


In [ ]:
# ==============================
# Merge Datasets Before Spliting
# ==============================
merged = pd.merge(
    prices,
    stocks,
    on='ticker',
    how='left'
)

print("Data After Merging")
print(merged.head())

In [ ]:
# ==============================
# Save cleaned and merged dataset
# ==============================
merged.to_csv('cleaned_stock_data_project1.csv')

print("Cleaned and Merged Dataset Saved Successfully")

In [ ]:
# =====================
# Split Data by Decade
# =====================
decades = dict(tuple(merged.groupby('Decade')))

print("Decades")
print(decades.keys())

In [ ]:
# ===================
# Summary Statistics
# ===================
for decade, df in decades.items():
    print(f"\nSummary Statistics for {decade}")
    print(df[['open','high','low','close','volume']].describe())


In [ ]:
# ======================
# Monthly Sector Trend
# ======================
for decade, df in decades.items():
print(f"\nMonthly Average Close Price by Sector ({decade})")
monthly_sector = df.groupby(['sector', pd.Grouper(freq='M')])['close'].mean()
print(monthly_sector.head())


In [ ]:
# =====================
# Volume Distribution
# =====================
for decade, df in decades.items():
print(f"\nVolume Distribution for {decade}")
df['volume'].hist(bins=50)
plt.title(f"{decade}s Volume Distribution")
plt.show()


In [ ]:
# =======================
# Price Spread (Box Plot)
# =======================
for decade, df in decades.items():
print(f"\nPrice Spread (High vs Low) for {decade}")
df[['high','low']].plot(kind='box')
plt.title(f"{decade}s Price Spread")
plt.show()


In [ ]:

# =================
# Sector Analysis
# =================

# Average Close Price
for decade, df in decades.items():
print(f"\nAverage Close Price by Sector ({decade}s))
print(df.groupby('sector')['close'].mean().sort_values(ascending=False))


# Volume by Sector
for decade, df in decades.items():
print(f"\nAverage Volume by Sector ({decade}s)")
print(df.groupby('sector')['volume'].mean().sort_values(ascending=False))


# Volatility
for decade, df in decades.items():
print(f"\nVolatility (Std Dev of Close) by Sector ({decade}s)")
print(df.groupby('sector')['close'].std().sort_values(ascending=False))
